In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
import gdown
from io import StringIO
from pathlib import Path
warnings.filterwarnings('ignore')

### Question 1: [IPO] Withdrawn IPOs by Company Type

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

url = "https://www.iposcoop.com/ipos-recently-filed/"
respond = requests.get(url, headers=headers)
respond.raise_for_status()

ipo = pd.read_html(StringIO(respond.text))[0]
ipo['Symbol'] = ipo['Symbol'].str.replace('.', '-', regex=False)
ipo['File Date'] = pd.to_datetime(ipo['File Date'], errors='coerce')

df = ipo.loc[
    (ipo['Expected To Trade'] == 'Withdrawn')
    & (ipo['File Date'] < '2026-09-11')
].copy()

company = df['Company'].fillna('')
df['Company Type'] = np.select(
    [
        company.str.contains(r'\bTechnologies\b', case=False, regex=True),
        company.str.contains(r'\bCorp(?:oration)?\b', case=False, regex=True),
        company.str.contains(r'\b(?:Inc|Incorporated)\b', case=False, regex=True),
        company.str.contains(r'\bGroup\b', case=False, regex=True),
        company.str.contains(r'\b(?:Ltd|Limited)\b', case=False, regex=True),
        company.str.contains(r'\bHoldings?\b', case=False, regex=True),
    ],
    ['Technologies', 'Acquisition Corp', 'Inc.', 'Group', 'Limited', 'Holdings'],
    default='Other',
)

def parse_numeric(value):
    if pd.isna(value) or str(value).strip() == '-':
        return np.nan
    return pd.to_numeric(str(value).replace('$', '').replace(',', '').strip(), errors='coerce')

for column in ['Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)']:
    df[column] = df[column].map(parse_numeric)

has_both_prices = df['Price Low'].notna() & df['Price High'].notna()
df['Avg_price'] = ((df['Price Low'] + df['Price High']) / 2).where(has_both_prices)
share_value = df['Shares (millions)'] * df['Avg_price']
df['Shares_offered_value'] = share_value.fillna(df['Est $ Vol (millions)'])

withdrawal_totals = (
    df.groupby('Company Type')['Shares_offered_value']
      .sum()
      .sort_values(ascending=False)
      .rename('Total withdrawn IPO value ($ millions)')
)
print(f"Withdrawn IPOs before 2026-09-11: {len(df)}")
display(withdrawal_totals.to_frame())
display(df[['Company', 'Company Type', 'Avg_price', 'Shares_offered_value']].head())

Withdrawn IPOs before 2026-09-11: 31


,Total withdrawn IPO value ($ millions)
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000


,Company,Company Type,Avg_price,Shares_offered_value
46,Motive Technologies (Withdrawn),Technologies,NaN,100.0
49,Idea Tech Holding (Withdrawn),Holdings,4.5,9.0
56,Hornbeck Offshore Services (Withdrawn),Other,NaN,100.0
58,Coolbit Technologies Ltd. (Withdrawn),Technologies,4.5,22.5
65,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,10.0,200.0


### Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [5]:
ipo_url = "https://www.iposcoop.com/2025-pricings/"
ipo_response = requests.get(ipo_url, headers=headers)
ipo_response.raise_for_status()
ipo_2025 = pd.read_html(StringIO(ipo_response.text))[0]

symbol_col = next(column for column in ipo_2025.columns if 'symbol' in str(column).lower() or 'ticker' in str(column).lower())
offer_date_col = next(column for column in ipo_2025.columns if 'offer date' in str(column).lower())
return_col = next(column for column in ipo_2025.columns if 'return' in str(column).lower())
ipo_2025[offer_date_col] = pd.to_datetime(ipo_2025[offer_date_col], errors='coerce')
ipo_2025['_return_numeric'] = pd.to_numeric(
    ipo_2025[return_col].astype('string').str.replace(r'[%,$+]', '', regex=True),
    errors='coerce',
)

filtered_ipos = ipo_2025.loc[
    (ipo_2025[offer_date_col] < '2025-09-01')
    & ipo_2025['_return_numeric'].notna()
    & (ipo_2025['_return_numeric'] != 0)
].copy()
tickers = filtered_ipos[symbol_col].dropna().astype(str).str.strip().unique().tolist()

raw_stocks = yf.download(
    tickers=tickers,
    start='2025-01-01',
    end='2026-09-12',
    group_by='ticker',
    auto_adjust=False,
    progress=False,
    threads=True,
)
if isinstance(raw_stocks.columns, pd.MultiIndex):
    field_level = next(
        level for level in range(raw_stocks.columns.nlevels)
        if 'Close' in raw_stocks.columns.get_level_values(level)
    )
    close = raw_stocks.xs('Close', axis=1, level=field_level)
else:
    close = raw_stocks[['Close']].copy()
    close.columns = tickers[:1]

if isinstance(close, pd.Series):
    close = close.to_frame(name=tickers[0])
close.columns = close.columns.astype(str)
close = close.dropna(axis=1, how='all').sort_index()

growth_252d = close / close.shift(252)
volatility = close.rolling(30).std() * np.sqrt(252)
sharpe = (growth_252d - 0.05).div(volatility.where(volatility.ne(0)))

feature_frames = {
    'Close': close,
    'growth_252d': growth_252d,
    'volatility': volatility,
    'Sharpe': sharpe,
}
stocks_df = pd.concat(
    [
        frame.rename_axis(index='Date', columns='Ticker').stack().rename(name)
        for name, frame in feature_frames.items()
    ],
    axis=1,
).reset_index()
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])

print(f"2025 IPO rows: {len(ipo_2025)}")
print(f"Filtered tickers: {len(tickers)}; tickers with Yahoo Finance data: {close.shape[1]}")
as_of_date = pd.Timestamp('2026-09-11')
as_of_df = stocks_df.loc[stocks_df['Date'].eq(as_of_date)]
print(f"Tickers with data on {as_of_date.date()}: {as_of_df['Ticker'].nunique()}")
display(as_of_df[['growth_252d', 'volatility', 'Sharpe']].describe())


15 Failed downloads:
['SKBL', 'AHL', 'TBH', 'CAEP', 'MTSR', 'MCTR', 'PTNM', 'WGRX', 'MJID', 'EMPG', 'EPWK', 'CEPT', 'AGH', 'SDM']: YFTzMissingError('possibly delisted; no timezone found')
['DLXY']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-12)')


2025 IPO rows: 231
Filtered tickers: 146; tickers with Yahoo Finance data: 131
Tickers with data on 2026-09-11: 131


,growth_252d,volatility,Sharpe
count,130.000000,129.000000,126.000000
mean,1.065371,27.996496,0.185804
std,3.048691,67.300098,0.580777
min,0.001005,0.000000,-0.040147
25%,0.150240,2.345137,0.010027
50%,0.603856,8.358907,0.045858
75%,1.056500,24.071310,0.120166
max,33.638270,627.698106,5.385326


### Question 3: [IPO] 'Fixed Months Holding Strategy'

In [6]:
stocks_df = stocks_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
future_growth_columns = []
for months in range(1, 13):
    column = f'future_growth_{months}_m'
    stocks_df[column] = (
        stocks_df.groupby('Ticker')['Close'].shift(-21 * months) / stocks_df['Close']
    )
    future_growth_columns.append(column)

first_trading_days = (
    stocks_df.dropna(subset=['Close'])
    .groupby('Ticker', as_index=False)
    .first()[['Ticker', 'Date', 'Close']]
    .rename(columns={'Date': 'min_date', 'Close': 'entry_close'})
)
entry_growth = first_trading_days.merge(
    stocks_df[['Ticker', 'Date', *future_growth_columns]],
    left_on=['Ticker', 'min_date'],
    right_on=['Ticker', 'Date'],
    how='inner',
).drop(columns='Date')

growth_stats = entry_growth[future_growth_columns].describe()
median_growth = entry_growth[future_growth_columns].median().dropna()
best_column = median_growth.idxmax()
best_month = int(best_column.removeprefix('future_growth_').removesuffix('_m'))
print(f"Tickers included: {len(entry_growth)}")
print(f"Highest median: month {best_month}, growth={median_growth[best_column]:.6f}x ({(median_growth[best_column] - 1) * 100:.2f}% return)")
display(growth_stats)

Tickers included: 131
Highest median: month 1, growth=0.930622x (-6.94% return)


,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,131.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,129.000000
mean,96.475125,68.493566,51.640993,22.870282,54.658381,64.501868,59.720109,21.620850,20.159934,11.342355,8.967293,5.863280
std,1092.037665,767.322940,575.469433,248.776601,611.660450,723.482471,668.665978,235.633979,216.992891,116.158189,90.379884,55.711436
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.709910,0.582405,0.449793,0.404543,0.431361,0.315944,0.254490,0.200089,0.180241,0.182353,0.169508,0.164831
50%,0.930622,0.891684,0.827951,0.736039,0.694830,0.713458,0.660986,0.603984,0.584421,0.533487,0.487132,0.508333
75%,1.114924,1.170970,1.126212,1.193000,1.091084,1.046022,1.018706,1.033314,1.022995,1.036785,1.077637,1.042121
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


### Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [10]:
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
data_path = Path('data.parquet')
if not data_path.exists():
    gdown.download(f"https://drive.google.com/uc?id={file_id}", str(data_path), quiet=False)

df = pd.read_parquet(data_path, engine='pyarrow')
if not any(str(column).lower() in {'date', 'datetime'} for column in df.columns):
    df = df.reset_index()

date_col = next(column for column in df.columns if str(column).lower() in {'date', 'datetime'})
rsi_col = next(column for column in df.columns if 'rsi' in str(column).lower())
forward_return_col = next(column for column in df.columns if str(column).lower() == 'growth_future_30d')
df[date_col] = pd.to_datetime(df[date_col], errors='coerce', utc=True).dt.tz_convert(None)

selected_df = df.loc[
    df[date_col].between('2000-01-01', '2025-06-01')
    & (pd.to_numeric(df[rsi_col], errors='coerce') < 30)
].dropna(subset=[forward_return_col]).copy()
net_income = 1000 * (selected_df[forward_return_col] - 1).sum()

print(f"Oversold trades: {len(selected_df):,}")
print(f"Average 30-day growth: {selected_df[forward_return_col].mean():.4f}")
print(f"Win rate: {(selected_df[forward_return_col] > 1).mean():.2%}")
print(f"Net income: ${net_income:,.2f} (${net_income / 1000:,.2f} thousand)")

Oversold trades: 5,206
Average 30-day growth: 1.0126
Win rate: 55.13%
Net income: $65,805.59 ($65.81 thousand)


### Question 5: Improving IPO Profitability

Rather than buying every IPO at its offer price, use a selective, rules-based approach:

- Screen for durable fundamentals: revenue growth, improving gross margins, manageable debt, positive or improving free cash flow, and enough cash runway.
- Compare valuation with established peers and avoid offerings priced for unusually optimistic growth.
- Treat float, lock-up expiry, insider ownership, underwriting quality, and institutional demand as risk factors, not standalone buy signals.
- Wait for price discovery (for example, require a base and a confirmed trend after the first few weeks) instead of assuming the first-day close is an attractive entry.
- Test explicit entry and exit rules with walk-forward, time-based out-of-sample validation; include slippage, fees, delistings, and a broad-market benchmark to avoid survivorship and look-ahead bias.
- Limit position size and diversify across sectors and IPO cohorts; define a maximum loss and an exit horizon before entering.

A sensible next step is to compare this screened, delayed-entry strategy with a passive IPO basket and the market index using the same historical periods.